# TP Jour 1 — Choix du Sujet Kaggle, Stockage Distribué HDFS & Couche Bronze

Bienvenue dans ce fil rouge **Data Lakehouse** ! Vous êtes libres de choisir votre thématique métier et votre jeu de données public sur **Kaggle**.

### Objectifs de la séance
1. **Cadrer votre sujet métier** et sélectionner un dataset Kaggle adapté au calcul distribué (~50 à 500 Mo).
2. **Explorer le cluster HDFS réel** (`1 NameNode` + `3 DataNodes`) configuré avec `dfs.blocksize = 16 Mo` et `dfs.replication = 2`.
3. **Ingérer vos fichiers bruts sur HDFS** et observer physiquement leur découpage en blocs de 16 Mo.
4. **Réaliser un crash-test en direct** : couper `datanode2` et observer la détection de panne + l'auto-réparation vers le 3e DataNode en ~50 secondes.
5. **Construire votre couche Bronze** (`/lakehouse/bronze/`) au format **Delta Lake** et inspecter votre premier job sur **Spark Web UI** (`http://localhost:4040`).

---
### Fiche de cadrage de votre projet *(À compléter par le groupe)*
- **Titre du sujet :** `Le prix des courses VTC à Boston : Uber vs Lyft (nov.–déc. 2018)`
- **Identifiant Kaggle (`auteur/nom-du-dataset`) :** `brllrb/uber-and-lyft-dataset-boston-ma`
- **Problématique métier principale :** `Qu'est-ce qui fait varier le prix d'une course VTC : l'heure, le trajet, la distance, la météo, la compagnie (Uber / Lyft), la gamme de véhicule ou la majoration « surge » ?`
- **Variables clés retenues (temporelles, quantitatives, catégorielles) :**
  - *Temporelles* : `datetime`, `timestamp`, `hour`, `day`, `month`
  - *Quantitatives* : `price`, `distance`, `surge_multiplier`, `temperature`, `precip_intensity`, `humidity`, `wind_speed`
  - *Catégorielles / géographiques* : `cab_type` (Uber / Lyft), `name` (gamme : Shared, UberX, Lux…), `source` et `destination` (12 quartiers de Boston), `short_summary` (météo)

## 1. Téléchargement de votre Dataset Kaggle dans `/workspace/data_raw`

Renseignez dans la cellule suivante l'identifiant Kaggle de votre dataset (ex. `"auteur/mon-dataset"`), ou déposez directement vos fichiers dans le dossier `data_raw/` à la racine du projet.
> **Critère important :** Vérifiez qu'au moins un fichier dépasse **20 à 32 Mo** afin d'observer sa division en plusieurs blocs HDFS de **16 Mo** !

In [1]:
import shutil
from pathlib import Path

DATA_RAW = Path("/workspace/data_raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)
SUPPORTED_EXTS = {".csv", ".parquet", ".json", ".jsonl", ".tsv"}

# TODO : Renseignez l'identifiant Kaggle de votre sujet (format "auteur/nom-du-dataset")
KAGGLE_DATASET = "brllrb/uber-and-lyft-dataset-boston-ma"  # Courses Uber & Lyft à Boston (nov.–déc. 2018)

local_files = sorted([f for f in DATA_RAW.iterdir() if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS])

if not local_files:
    assert KAGGLE_DATASET.strip(), (
        "TODO : Renseignez `KAGGLE_DATASET = 'auteur/nom-du-dataset'` (ou déposez vos fichiers dans data_raw/)."
    )
    import kagglehub
    print(f"Téléchargement de {KAGGLE_DATASET} depuis Kaggle Hub...")
    cache_dir = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    for src_file in sorted(cache_dir.rglob("*")):
        if src_file.is_file() and src_file.suffix.lower() in SUPPORTED_EXTS:
            shutil.copy2(src_file, DATA_RAW / src_file.name)
    local_files = sorted([f for f in DATA_RAW.iterdir() if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS])

assert len(local_files) > 0, "Aucun fichier de données détecté dans /workspace/data_raw/."

print(f"Inventaire local dans {DATA_RAW} ({len(local_files)} fichier(s)) :")
for f in local_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    expected_blocks = max(1, int(-(-f.stat().st_size // (16 * 1024 * 1024))))
    print(f" - {f.name:<45} {size_mb:>7.2f} Mo  (~{expected_blocks} bloc(s) HDFS de 16 Mo)")

Inventaire local dans /workspace/data_raw (1 fichier(s)) :
 - rideshare_kaggle.csv                           350.36 Mo  (~22 bloc(s) HDFS de 16 Mo)


## 2. Démarrage de la `SparkSession` pré-configurée sur HDFS

Grâce au fichier `config/spark-defaults.conf`, Spark pointe automatiquement vers `hdfs://namenode:9000` avec **Delta Lake** activé.

### Exercice 2.1 — Initialiser `SparkSession` et contrôler la configuration HDFS
1. Créez une session Spark nommée `"J1_HDFS_Ingestion_Bronze"`.
2. Récupérez via `spark.conf.get(...)` la valeur de `"spark.hadoop.fs.defaultFS"`, `"spark.hadoop.dfs.blocksize"` et `"spark.hadoop.dfs.replication"`.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# La config HDFS + Delta Lake est chargée automatiquement depuis config/spark-defaults.conf
spark = SparkSession.builder.appName("J1_HDFS_Ingestion_Bronze").getOrCreate()

assert spark is not None, "TODO : Initialisez `spark` avec SparkSession.builder..."

# Les 3 paramètres HDFS lus depuis la configuration de la session
default_fs = spark.conf.get("spark.hadoop.fs.defaultFS")
block_size = spark.conf.get("spark.hadoop.dfs.blocksize")
replication = spark.conf.get("spark.hadoop.dfs.replication")

print("Spark version       :", spark.version)
print("Default FileSystem  :", default_fs)
print("HDFS Block Size (o) :", block_size)
print("HDFS Replication    :", replication)

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Spark version       : 3.5.3
Default FileSystem  : hdfs://namenode:9000
HDFS Block Size (o) : 16777216
HDFS Replication    : 2


## 3. Copie des fichiers bruts vers HDFS & Inspection des Blocs de 16 Mo

La cellule suivante téléverse vos fichiers bruts de `/workspace/data_raw` vers `/lakehouse/bronze/raw/` sur HDFS, puis interroge le NameNode pour afficher la localisation exacte de chaque bloc de 16 Mo sur les 3 DataNodes (`datanode1`, `datanode2`, `datanode3`).

In [3]:
jvm = spark._jvm
hadoop_conf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
PathHDFS = jvm.org.apache.hadoop.fs.Path

raw_hdfs_dir = PathHDFS("/lakehouse/bronze/raw")
fs.mkdirs(raw_hdfs_dir)

for file_path in local_files:
    src = PathHDFS(f"file://{file_path}")
    dst = PathHDFS(f"/lakehouse/bronze/raw/{file_path.name}")
    fs.copyFromLocalFile(False, True, src, dst)
    print(f"Téléversé vers HDFS : {dst}")

Téléversé vers HDFS : /lakehouse/bronze/raw/rideshare_kaggle.csv


In [4]:
# Inspection physique du découpage en blocs de 16 Mo et identification du plus gros fichier pour le crash-test
statuses = sorted(fs.listStatus(raw_hdfs_dir), key=lambda s: s.getLen(), reverse=True)
largest_hdfs_path = statuses[0].getPath().toString()

for status in statuses:
    path = status.getPath()
    length_mb = status.getLen() / (1024 * 1024)
    blocks = fs.getFileBlockLocations(status, 0, status.getLen())
    print(f"Fichier : {path.getName()} ({length_mb:.2f} Mo | repl={status.getReplication()})")
    for idx, blk in enumerate(blocks):
        print(f"   Bloc #{idx} ({blk.getLength() / (1024*1024):5.2f} Mo) -> DataNodes : {list(blk.getHosts())}")

Fichier : rideshare_kaggle.csv (350.36 Mo | repl=2)
   Bloc #0 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode1']
   Bloc #1 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode1']
   Bloc #2 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #3 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode2']
   Bloc #4 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #5 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode1']
   Bloc #6 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode3']
   Bloc #7 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode1']
   Bloc #8 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode1']
   Bloc #9 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #10 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #11 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode2']
   Bloc #12 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode3']
   Bloc #13 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode2']
   Bloc #14 (16.00 Mo) -> DataNodes : ['datanode

### Questions d'analyse (à vérifier aussi sur http://localhost:9870)
1. Comparez sur l'interface Web du NameNode (`Utilities > Browse the file system`) le nombre de blocs de votre fichier le plus volumineux avec sa taille divisée par 16 Mo.
2. Un fichier de 3 Mo bloque-t-il réellement 16 Mo d'espace disque sur les DataNodes ? Pourquoi ?
3. Pourquoi avoir configuré `dfs.replication = 2` sur un cluster à `3` DataNodes ?

### Réponses aux questions d'analyse

**1. Nombre de blocs du plus gros fichier.**
`rideshare_kaggle.csv` pèse **350,36 Mo**. 350,36 / 16 = 21,9, donc arrondi au supérieur **22 blocs** : 21 blocs pleins de 16 Mo + 1 dernier bloc de **14,36 Mo**. C'est bien ce qu'affichent la cellule ci-dessus, `hdfs fsck` (« Total blocks : 22 ») et `Utilities > Browse the file system` sur http://localhost:9870 (Block 0 à Block 21).

**2. Un fichier de 3 Mo bloque-t-il 16 Mo de disque ?**
Non. 16 Mo est une taille **maximale** de bloc, pas une réservation : chaque bloc est stocké comme un fichier ordinaire sur le disque du DataNode, à sa taille réelle. Un fichier de 3 Mo occupe donc 3 Mo par copie (6 Mo au total avec la réplication 2). On le voit avec notre dernier bloc, qui fait 14,36 Mo et non 16. Le vrai coût d'un petit fichier est ailleurs : chaque fichier et chaque bloc prend de la place dans la **mémoire du NameNode** (environ 150 octets par objet). Des millions de petits fichiers saturent donc le NameNode : c'est le « small files problem ».

**3. Pourquoi `dfs.replication = 2` sur 3 DataNodes ?**
- Chaque bloc existe en 2 exemplaires sur 2 machines différentes : on peut **perdre une machine sans perdre de donnée**.
- Il reste toujours **une 3e machine libre** pour recréer la copie manquante après une panne. C'est exactement ce qu'on observe au crash-test. Avec une réplication de 3 sur 3 DataNodes, chaque machine aurait déjà tout : après une panne, impossible de revenir à 3 copies, et les blocs resteraient sous-répliqués.
- On consomme 2 fois l'espace disque au lieu de 3, ce qui compte sur un PC portable.

---
## 4. Crash-Test en Direct : Panne d'un DataNode et Auto-Réparation (~50s)

Dans `config/hdfs-site.xml`, le délai de détection d'un DataNode mort a été abaissé de 10m30s à **~50 secondes** :
$$2 \times \text{recheck-interval (10s)} + 10 \times \text{heartbeat.interval (3s)} = 50\text{ s}$$

### Protocole à réaliser depuis votre terminal :
1. Inspectez l'état initial des blocs : `task hdfs:fsck`
2. Arrêtez brutalement le DataNode 2 :
   ```bash
   task hdfs:crash-dn2   # équivalent à : docker stop datanode2
   ```
3. Observez l'onglet **Datanodes** sur `http://localhost:9870` pendant ~50s : dès que `datanode2` passe à **Dead**, le NameNode détecte les blocs sous-répliqués et ordonne leur copie automatique vers le 3e DataNode survivant !

### Exercice 4.1 — Vérifier la continuité de service et la *Data Locality*
Pendant que `datanode2` est éteint, lisez votre plus gros fichier (`largest_hdfs_path`) avec PySpark, affichez `df_check.count()` ainsi que `df_check.rdd.getNumPartitions()`, et expliquez le lien avec le nombre de blocs HDFS.

*(Pensez ensuite à rallumer le nœud avec `task hdfs:restore-dn2` !)*

In [5]:
print("Fichier cible du test de résilience :", largest_hdfs_path)

# Lecture du CSV pendant que datanode2 est arrêté : HDFS sert les blocs depuis les copies survivantes
df_check = spark.read.option("header", "true").csv(largest_hdfs_path)

assert df_check is not None, "TODO : Lisez `largest_hdfs_path` dans `df_check`."

print(f"Nombre total de lignes lues       : {df_check.count():,}")
print(f"Nombre de partitions Spark créées : {df_check.rdd.getNumPartitions()}")

# Bonus : en alignant la taille max d'une partition Spark sur la taille d'un bloc HDFS (16 Mo),
# on retrouve « 1 bloc HDFS = 1 partition Spark »
spark.conf.set("spark.sql.files.maxPartitionBytes", block_size)
df_blocks = spark.read.option("header", "true").csv(largest_hdfs_path)
print(f"Partitions avec maxPartitionBytes = 16 Mo : {df_blocks.rdd.getNumPartitions()}")
spark.conf.unset("spark.sql.files.maxPartitionBytes")

Fichier cible du test de résilience : hdfs://namenode:9000/lakehouse/bronze/raw/rideshare_kaggle.csv


Nombre total de lignes lues       : 693,071
Nombre de partitions Spark créées : 22


Partitions avec maxPartitionBytes = 16 Mo : 22


### Observations du crash-test et réponse à l'exercice 4.1

| Étape | Mesure |
|---|---|
| État initial (`task hdfs:fsck`) | 22 blocs, réplication moyenne 2.0, statut **HEALTHY** ; **16 blocs sur 22** avaient une copie sur `datanode2` |
| `task hdfs:crash-dn2` | `datanode2` déclaré **Dead** sur http://localhost:9870 après **~53 s** (théorie : 2 × 10 s + 10 × 3 s = 50 s, mesuré toutes les 3 s) |
| Juste après la détection | **0 bloc manquant** ; le NameNode lance la recopie, il ne reste plus que 6 blocs sous-répliqués (réplication moyenne 1,73) |
| Quelques secondes plus tard | **0 bloc sous-répliqué**, réplication revenue à 2.0 : `datanode1` et `datanode3` portent chacun les 22 blocs (353 Mo) |
| `task hdfs:restore-dn2` | `datanode2` revenu dans le cluster en ~6 s (3 DataNodes vivants) |

**Avertissements pendant l'écriture Bronze :** juste après le retour de `datanode2`, certains blocs avaient **3 copies** (les 2 recréées pendant la panne + l'ancienne copie restée sur `datanode2`). Le NameNode a supprimé les copies en trop pendant que Spark lisait le fichier : Spark a affiché des avertissements `Replica not found`, puis il a relu chaque bloc concerné sur une autre copie. **Aucune donnée perdue** (693 071 lignes en Bronze). Ces journaux techniques (avertissements Java) ont été retirés du notebook pour le rendre lisible.

**Continuité de service :** pendant que `datanode2` était éteint, Spark a lu **693 071 lignes** sans aucune erreur. Pour chaque bloc, le client HDFS va chercher la copie survivante sur `datanode1` ou `datanode3`.

**Lien partitions / blocs :** Spark a créé **22 partitions** pour **22 blocs HDFS** : une tâche Spark par bloc. Sur un vrai cluster, chaque tâche s'exécute sur la machine qui stocke son bloc (**Data Locality** : on déplace le calcul vers la donnée, pas la donnée vers le calcul). Ici, Spark tourne en `local[*]` dans le conteneur `spark-jupyter`, donc la localité n'est pas exploitée physiquement, mais le découpage est le même.

*Nuance :* Spark ne découpe pas directement par bloc HDFS, mais selon `spark.sql.files.maxPartitionBytes` (128 Mo par défaut) et le nombre de cœurs. Le conteneur a 22 cœurs, donc 367 Mo / 22 ≈ 16 Mo par partition : l'égalité est en partie une coïncidence. En fixant `maxPartitionBytes` à 16 Mo (la taille d'un bloc), on **garantit** une partition par bloc : on obtient bien 22 partitions (dernière ligne de la cellule).

---
## 5. Construction de votre Couche Bronze en Delta Lake (`/lakehouse/bronze/`)

### Exercice 5.1 — Convertir vos fichiers bruts en tables Delta Lake Bronze
Dans une architecture **Médaillon**, la couche **Bronze** conserve les données brutes fidèles à la source tout en les stockant au format transactionnel **Delta Lake** enrichi de métadonnées de traçabilité :
- `_ingested_at` : `F.current_timestamp()`
- `_source_file` : `F.input_file_name()`

> **Attention aux noms de colonnes Delta/Parquet :** Si vos fichiers CSV bruts contiennent des espaces ou caractères spéciaux dans les en-têtes, normalisez-les simplement en `snake_case` (ex. `.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])`).

Complétez la cellule ci-dessous pour ingérer chacune des tables utiles de votre sujet dans `/lakehouse/bronze/<nom_table>`.

In [6]:
import re

# TODO : Définissez le dictionnaire { "nom_table_bronze": "/lakehouse/bronze/raw/votre_fichier.csv", ... }
BRONZE_SOURCES: dict[str, str] = {
    "rides": "/lakehouse/bronze/raw/rideshare_kaggle.csv",  # 1 ligne = 1 course Uber ou Lyft
}

assert len(BRONZE_SOURCES) > 0, "TODO : Renseignez au moins une table à ingérer dans `BRONZE_SOURCES`."

for table_name, raw_path in BRONZE_SOURCES.items():
    bronze_delta_path = f"/lakehouse/bronze/{table_name}"

    # 1. Lecture du CSV brut (en-tête + typage automatique des colonnes)
    df_raw = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(raw_path)
    )

    assert df_raw is not None, f"TODO : Construisez et écrivez la table Bronze `{table_name}`."

    # 2. Noms de colonnes en snake_case : « apparentTemperature » -> « apparent_temperature »,
    #    « visibility.1 » -> « visibility_1 » (le point est interdit dans un nom de colonne Delta)
    df_raw = df_raw.toDF(*[
        re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", c.strip()).lower().replace(" ", "_").replace(".", "_")
        for c in df_raw.columns
    ])

    # 3. Métadonnées de traçabilité
    df_raw = (
        df_raw
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.input_file_name())
    )

    # 4. Écriture au format Delta Lake (table transactionnelle) sur HDFS
    (
        df_raw.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(bronze_delta_path)
    )

    cnt = spark.read.format("delta").load(bronze_delta_path).count()
    print(f"[BRONZE OK] {bronze_delta_path:<38} -> {cnt:>10,} lignes")

[BRONZE OK] /lakehouse/bronze/rides                ->    693,071 lignes


In [7]:
# Auto-contrôle de fin de Jour 1 : au moins une table Delta valide dans /lakehouse/bronze/
bronze_root = PathHDFS("/lakehouse/bronze")
bronze_delta_tables = [
    s.getPath().getName()
    for s in fs.listStatus(bronze_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(bronze_delta_tables) > 0, "Aucune table Delta détectée dans /lakehouse/bronze/ !"
print("Tables Delta Bronze validées :", bronze_delta_tables)

Tables Delta Bronze validées : ['rides']
